## 02 Datenbereinigung und Vorbereitung

Author: "Zhanna Davtyan"

***Hinweis:*** Führen Sie dieses Notebook aus, nachdem Sie das `01 Daten Laden` Notebook ausgeführt haben, um die erforderlichen Metadaten zu generieren.

Der Rohdatensatz umfasst eine Vielzahl an Spalten, von denen viele für die Analyse irrelevant oder unvollständig sind.

In [ ]:
import numpy as np
import os
import pandas as pd

from pathlib import Path

In [ ]:
base_dir = Path.cwd()
if base_dir.name == "notebooks":
    base_dir = base_dir.parent
    os.chdir(base_dir)


In [ ]:
df = pd.read_pickle("ergebnisse/instagram_node_dataframe.pkl")
df.head() 

### Datenaufbereitung für die Analyse

Es werden für die Analyse relevante Daten aufbereitet und passendere Spaltennamen vergeben.

In [ ]:
df_gefiltert = df.rename(columns={
    "edge_media_preview_like.count": "likes",
    "owner.edge_followed_by.count": "follower_count",
    "owner.is_verified": "is_verified",
    "owner.category_name": "account_category",
    "owner.is_business_account": "is_business_account",
    "owner.is_professional_account": "is_professional_account",
    "edge_media_to_comment.count": "comment_count",
    "edge_media_to_tagged_user.edges": "tagged_users",
    "edge_media_to_caption.edges": "caption_text",
    "__typename": "post_type",
    "edge_sidecar_to_children.edges": "is_carousel",
    "taken_at_timestamp": "timestamp",
    "folder_name": "post_id"
    })

df_gefiltert = df_gefiltert[[
    "likes",
    "follower_count",
    "is_verified",
    "account_category",
    "is_business_account",
    "is_professional_account",
    "comment_count",
    "tagged_users",
    "caption_text",
    "post_type",
    "is_carousel",
    "timestamp",   
    "post_id"
]]
df_gefiltert.head()

In [ ]:
df_gefiltert.info()

### Vorbereitung der Zielvariable

Die Rohdaten der "Likes" werden optimiert:

- Negative Werte auf 0 setzen
- Log-Transformation für normalere Verteilung:

``` python
np.log1p(x) ≡ np.log(1 + x) # 1+ x, um log(0)= -∞ zu vermeiden
```

In [ ]:
df_gefiltert['likes'] = df_gefiltert['likes'].clip(lower=0).astype(int)
# Log-Transformation der Likes
df_gefiltert['likes_log'] = np.log1p(df_gefiltert['likes'])
print("\n'likes_log' Spalte erstellt.")
df_gefiltert[['likes_log', 'likes']]

#### Feature-Engineering

##### Zeitbasierte Merkmale: 

Aus dem Timestamp wurden Stunde des Tages, Wochentag und Monat abgeleitet.

In [ ]:
df_gefiltert['datetime'] = pd.to_datetime(df_gefiltert['timestamp'], unit='s')
df_gefiltert['hour_of_day'] = df_gefiltert['datetime'].dt.hour
df_gefiltert['day_of_week'] = df_gefiltert['datetime'].dt.day_of_week + 1
df_gefiltert['month'] = df_gefiltert['datetime'].dt.month
      
print("\nZeitbasierte Merkmale erstellt.")
df_gefiltert[['datetime', 'hour_of_day', 'day_of_week', 'month']].head()

#### Text-Features: 

Aus den Bild-Beschreibungen wurden Merkmale wie Text-Länge, Hashtag-Anzahl und Erwähnungen extrahiert.

In [ ]:
def extract_caption(edges):
    if isinstance(edges, list) and len(edges) > 0 and 'node' in edges[0] and 'text' in edges[0]['node']:
        return edges[0]['node']['text']
    return "" 

df_gefiltert['actual_caption_text'] = df_gefiltert['caption_text'].apply(extract_caption)
df_gefiltert['caption_length'] = df_gefiltert['actual_caption_text'].apply(len)
df_gefiltert['hashtag_count'] = df_gefiltert['actual_caption_text'].str.count('#')
df_gefiltert['mention_count'] = df_gefiltert['actual_caption_text'].str.count('@')

df_gefiltert[['actual_caption_text', 'caption_length', 'hashtag_count', 'mention_count']].head()

#### Medien-Feature: 

Information zu Karussell-Posts (mehrere Bilder) wurde als binäres Feature kodiert.

In [ ]:
def check_carousel_and_count(edges):
    if isinstance(edges, list) and len(edges) > 0:
        return True, len(edges) # Ist Karussell, Anzahl der Medien
    return False, 1 # Kein Karussell (oder einzelnes Medium)

results = df_gefiltert['is_carousel'].apply(lambda x: check_carousel_and_count(x)) 
df_gefiltert['is_carousel_bool'] = [res[0] for res in results]
df_gefiltert['media_count_in_post'] = [res[1] for res in results]

# Für Nicht-Karussell-Posts ('GraphImage', (kein 'GraphSidecar/Karusell')) ist media_count_in_post typischerweise 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'media_count_in_post'] = 1
df_gefiltert.loc[df_gefiltert['post_type'] != 'GraphSidecar', 'is_carousel_bool'] = False

df_gefiltert[['is_carousel_bool', 'media_count_in_post']].head()

In [ ]:
df_gefiltert.head()

In [ ]:
# DataFrame im "ergebnisse"-Ordner speichern
df_gefiltert.to_pickle("ergebnisse/verarbeitete_daten.pkl")